### <div style="background-color:teal; color:white; padding:10px;"> Data Loading & Multi-Hot Targets </div>

In [1]:
# ===== CONFIG =====
from pathlib import Path
CH_ROOT  = Path(r'D:/Datasets/Datasets/Charades')
RGB_ROOT = CH_ROOT/'RGB'          # RGB/<video>/rgb/frame_%06d.jpg
FLOW_ROOT= CH_ROOT/'Flow'         # Flow/<video>/flow/frame_%06d.jpg
ANNO_CSV = CH_ROOT/'Label'/'Charadesh_Annotation.csv'
OUT_DIR  = Path('./artifacts_charades'); OUT_DIR.mkdir(exist_ok=True)   # separate from ADL

VIDEOS   = None            # None = all video_ids present in the CSV; or an explicit list
N            = 60          # frames per window (Charades videos are short -> smaller N; sweep in NB03)
STRIDE       = N
INCLUDE_BACKGROUND = False
SEED_FRAC    = 0.10        # default; NB03 can sweep 5/10/25%
MIN_PER_CLASS= 2
RANDOM_SEED  = 42
print('Charades NB01 config OK')

Charades NB01 config OK


In [2]:
import numpy as np, pandas as pd
rng=np.random.default_rng(RANDOM_SEED)
def rgb_dir(v):  return RGB_ROOT /v/'rgb'
def flow_dir(v): return FLOW_ROOT/v/'flow'
def fname(i):    return f'frame_{i+1:06d}.jpg'   # internal 0-based idx i -> 1-indexed 6-digit file

# read CSV (utf-8-sig strips the BOM on video_id)
DF=pd.read_csv(ANNO_CSV, encoding='utf-8-sig')
DF.columns=[c.strip() for c in DF.columns]
print('columns:', list(DF.columns))
if VIDEOS is None: VIDEOS=sorted(DF['video_id'].unique().tolist())
print(f'{len(VIDEOS)} videos:', VIDEOS)

columns: ['video_id', 'StartFrame', 'EndFrame', 'Action_class', 'Action_name']
9 videos: ['00HFP', '00MFE', '00N38', '00NN7', '00SL4', '00T4B', '00X3U', '00YZL', '00ZCA']


### <div style="background-color:teal; color:white; padding:10px;"> 1. Global class map (over ALL videos in the CSV) </div>

In [3]:
alla=DF[['Action_class','Action_name']].drop_duplicates('Action_class').sort_values('Action_class')
id2name=dict(zip(alla.Action_class,alla.Action_name)); ids=sorted(id2name)
off=1 if INCLUDE_BACKGROUND else 0
labelid2cls={l:i+off for i,l in enumerate(ids)}
names={0:'background'} if INCLUDE_BACKGROUND else {}
for l,ci in labelid2cls.items(): names[ci]=id2name[l]
CMAP=dict(labelid2cls=labelid2cls,names=names,C=len(ids)+off)
print('num classes C =',CMAP['C'])

num classes C = 39


### <div style="background-color:teal; color:white; padding:10px;"> 2. Per-frame multi-hot targets (StartFrame/EndFrame are 1-indexed inclusive) </div>

In [4]:
def num_frames(v):
    return len(sorted(rgb_dir(v).glob('frame_*.jpg')))
def build_multihot(v,nf,cmap):
    sub=DF[DF['video_id']==v]; Y=np.zeros((nf,cmap['C']),np.float32)
    for _,r in sub.iterrows():
        ci=cmap['labelid2cls'][int(r.Action_class)]
        s=max(0,int(r.StartFrame)-1); e=min(int(r.EndFrame),nf)   # 1-indexed inclusive -> [s:e]
        Y[s:e,ci]=1.0
    if INCLUDE_BACKGROUND: Y[Y[:,1:].sum(1)==0,0]=1.0
    return Y
TARGETS={}
for v in VIDEOS:
    if not rgb_dir(v).exists(): raise FileNotFoundError(f'RGB missing: {rgb_dir(v)}')
    nf=num_frames(v); TARGETS[v]=build_multihot(v,nf,CMAP); Y=TARGETS[v]
    print(f'{v}: frames={nf} | labeled={int((Y.sum(1)>0).sum())} | multi(>=2)={int((Y.sum(1)>=2).sum())}')

00HFP: frames=746 | labeled=579 | multi(>=2)=212
00MFE: frames=493 | labeled=493 | multi(>=2)=352
00N38: frames=588 | labeled=503 | multi(>=2)=445
00NN7: frames=735 | labeled=722 | multi(>=2)=611
00SL4: frames=215 | labeled=215 | multi(>=2)=207
00T4B: frames=484 | labeled=484 | multi(>=2)=8
00X3U: frames=506 | labeled=479 | multi(>=2)=278
00YZL: frames=796 | labeled=781 | multi(>=2)=781
00ZCA: frames=2229 | labeled=941 | multi(>=2)=840


### <div style="background-color:teal; color:white; padding:10px;"> 3. Windowing + stratified seeds </div>

In [5]:
def make_windows(nf,N,stride): return [list(range(s,s+N)) for s in range(0,max(nf-N+1,1),stride)] if nf>=N else [list(range(nf))]
def stratified_seeds(Y,frac,m,rng):
    n,C=Y.shape; labeled=np.where(Y.sum(1)>0)[0]; budget=max(int(round(frac*len(labeled))),1); chosen=set()
    for c in range(C):
        pool=np.where(Y[:,c]>0)[0]
        if len(pool): chosen.update(rng.choice(pool,min(m,len(pool)),replace=False).tolist())
    rem=[i for i in labeled if i not in chosen]
    if len(chosen)<budget and rem: chosen.update(rng.choice(rem,min(budget-len(chosen),len(rem)),replace=False).tolist())
    mask=np.zeros(n,bool); mask[list(chosen)]=True; return mask
WINDOWS={v:make_windows(TARGETS[v].shape[0],N,STRIDE) for v in VIDEOS}
SEEDS={v:stratified_seeds(TARGETS[v],SEED_FRAC,MIN_PER_CLASS,rng) for v in VIDEOS}
for v in VIDEOS:
    print(f'{v}: {len(WINDOWS[v])} windows(N={N}) | seeds={int(SEEDS[v].sum())}/{int((TARGETS[v].sum(1)>0).sum())}')

00HFP: 12 windows(N=60) | seeds=58/579
00MFE: 8 windows(N=60) | seeds=49/493
00N38: 9 windows(N=60) | seeds=50/503
00NN7: 12 windows(N=60) | seeds=72/722
00SL4: 3 windows(N=60) | seeds=22/215
00T4B: 8 windows(N=60) | seeds=48/484
00X3U: 8 windows(N=60) | seeds=48/479
00YZL: 13 windows(N=60) | seeds=78/781
00ZCA: 37 windows(N=60) | seeds=94/941


### <div style="background-color:teal; color:white; padding:10px;"> 4. Class coverage (per video + pooled) — rare-class check </div>

In [6]:
names=CMAP['names']; pooled_seed=np.zeros(CMAP['C']); pooled_lab=np.zeros(CMAP['C'])
per_seed={v:TARGETS[v][SEEDS[v]].sum(0) for v in VIDEOS}; per_lab={v:(TARGETS[v]>0).sum(0) for v in VIDEOS}
for ci in range(CMAP['C']):
    ps=sum(per_seed[v][ci] for v in VIDEOS); pl=sum(per_lab[v][ci] for v in VIDEOS)
    pooled_seed[ci]=ps; pooled_lab[ci]=pl
present=int((pooled_lab>0).sum()); zero=[names[ci] for ci in range(CMAP['C']) if pooled_lab[ci]>0 and pooled_seed[ci]==0]
print(f'classes present: {present}/{CMAP["C"]}')
print(f'present classes with ZERO pooled seeds: {zero if zero else "none"}')
print('per-class pooled (seed / labeled):')
for ci in range(CMAP['C']):
    if pooled_lab[ci]>0: print(f'  {names[ci][:34]:<36} seed={int(pooled_seed[ci]):>4}  lab={int(pooled_lab[ci]):>6}')

classes present: 37/39
present classes with ZERO pooled seeds: none
per-class pooled (seed / labeled):
  Holding some clothes                 seed=  28  lab=   319
  Putting clothes somewhere            seed=  48  lab=   395
  Closing a door                       seed=  60  lab=   608
  Opening a door                       seed=  45  lab=   383
  Working at a table                   seed=  19  lab=   217
  Holding a bag                        seed=  26  lab=   232
  Working/Playing on a laptop          seed=  21  lab=   250
  Holding a shoe/shoes                 seed=  48  lab=   471
  Putting on shoe/shoes                seed=  25  lab=   224
  Throwing shoes somewhere             seed=  19  lab=   178
  Sitting in a chair                   seed=  27  lab=   315
  Eating a sandwich                    seed=  61  lab=   611
  Holding a sandwich                   seed=  72  lab=   731
  Holding a blanket                    seed=  34  lab=   263
  Taking a blanket from somewhere      seed

### <div style="background-color:teal; color:white; padding:10px;"> 5. Save (to artifacts_charades) </div>

In [7]:
import pickle
pickle.dump(CMAP,open(OUT_DIR/'class_map.pkl','wb'))
for v in VIDEOS:
    np.savez_compressed(OUT_DIR/f'{v}_data.npz',targets=TARGETS[v],seeds=SEEDS[v],windows=np.array(WINDOWS[v],dtype=object))
print('saved',len(VIDEOS),'videos ->',OUT_DIR.resolve())

saved 9 videos -> C:\Users\PAWANESH\ICVGIP\Charades\artifacts_charades
